# Prepare data

## 0. Setup

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
ORIGIN_DIR = PROJECT_DIR / "0_origin"
PROCESSED_DIR = PROJECT_DIR / "1_processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Station data

In [15]:
df_stations = pd.read_csv(ORIGIN_DIR / "Station_registry_information.CSV", header=0, index_col="AirQualityStation")
print(f"Total number of stations: {df_stations.shape[0]}; features: {df_stations.shape[1]}")
df_stations.head()

Total number of stations: 744; features: 14


,Longitude,Latitude,Altitude,AirQualityStationType,AirQualityStationArea,COD_RIP,COD_REG,COD_PROV,COD_CM,COD_UTS,PRO_COM,PRO_COM_T,COMUNE,Area comune
AirQualityStation,,,,,,,,,,,,,,
STA.IT0063A,10.534167,42.939167,37.0,industrial,suburban,3,9,49,0,49,49012,49012,Piombino,1.307445e+08
STA.IT0187A,11.613060,44.842500,8.0,traffic,urban,2,8,38,0,38,38008,38008,Ferrara,4.055220e+08
STA.IT0267A,9.414722,45.329444,80.0,background,suburban,1,3,98,0,98,98056,98056,Tavazzano con Villavesco,1.606860e+07
STA.IT0448A,12.313889,45.429444,1.0,background,urban,2,5,27,227,227,27042,27042,Venezia,4.183949e+08
STA.IT0459A,13.341940,43.598890,15.0,background,suburban,3,11,42,0,42,42014,42014,Chiaravalle,1.760196e+07


Add province, region and area name

In [16]:
df_stations = df_stations.rename(columns={"COMUNE": "Comune"})

df_comuni = pd.read_excel(
    ORIGIN_DIR / "Elenco-comuni-italiani.xlsx",
    sheet_name=0,
    usecols=[5, 9, 10, 11],
)
df_comuni.columns = [
    "Comune",
    "Ripartizione geografica",
    "Regione",
    "Provincia",
]
df_comuni = df_comuni.loc[df_comuni["Comune"].isin(df_stations["Comune"])]

df_stations = df_stations.join(
    df_comuni.set_index("Comune"),
    on="Comune",
    how="left",
    validate="many_to_one",
)
print(
    "Stations without a municipality match: "
    f"{df_stations['Ripartizione geografica'].isna().sum()}"
)

Stations without a municipality match: 0


Remove useless columns and save

In [17]:
df_stations = df_stations[["AirQualityStationType", "AirQualityStationArea",
                          "Comune", "Provincia", "Regione", "Ripartizione geografica",
                          "Longitude", "Latitude", "Altitude"]].copy()
df_stations.to_csv(PROCESSED_DIR / "station_info.csv")
df_stations.head()


,AirQualityStationType,AirQualityStationArea,Comune,Provincia,Regione,Ripartizione geografica,Longitude,Latitude,Altitude
AirQualityStation,,,,,,,,,
STA.IT0063A,industrial,suburban,Piombino,Livorno,Toscana,Centro,10.534167,42.939167,37.0
STA.IT0187A,traffic,urban,Ferrara,Ferrara,Emilia-Romagna,Nord-est,11.613060,44.842500,8.0
STA.IT0267A,background,suburban,Tavazzano con Villavesco,Lodi,Lombardia,Nord-ovest,9.414722,45.329444,80.0
STA.IT0448A,background,urban,Venezia,Venezia,Veneto,Nord-est,12.313889,45.429444,1.0
STA.IT0459A,background,suburban,Chiaravalle,Ancona,Marche,Centro,13.341940,43.598890,15.0


## 2. Measurements

In [18]:
import gc

measurement_files = sorted(ORIGIN_DIR.glob("GRINS_AQCLIM_points*.csv"))
if not measurement_files:
    raise FileNotFoundError(f"No GRINS_AQCLIM_points CSV files found in {ORIGIN_DIR}")

df_measurement_parts = [pd.read_csv(file) for file in measurement_files]
df_measurements = pd.concat(df_measurement_parts, ignore_index=True, sort=False)
del df_measurement_parts
gc.collect()

print(
    f"Loaded {len(measurement_files)} files into df_measurements "
    f"({df_measurements.shape[0]} rows, {df_measurements.shape[1]} columns)."
)
df_measurements.to_csv(PROCESSED_DIR / "raw.csv")

Loaded 6 files into df_measurements (2988648 rows, 59 columns).


## 3. Daily indices

Indices are computed **per station-day** from the daily statistics, using the statistic that matches each definition. NH₃ is excluded (one station); CO is handled in mg/m³.

| Index | Statistic used |
|---|---|
| `NOx` (NO₂-equivalent), `NO2_NOx`, `Ox_ppb`, `RPM` (PM₂.₅/PM₁₀) | daily **mean** |
| 24-h limits and WHO 2021 guidelines (PM₁₀, PM₂.₅, NO₂, SO₂, CO) | daily **mean** |
| Hourly thresholds (NO₂ 200, SO₂ 350, O₃ 180/240) | daily **max** |
| `AQI_lo` / `AQI_hi` (EEA-inspired, class 1–6) | lo: all means; hi: PM on mean, NO₂/O₃/SO₂ on max (worst hour) |
| `AQHI_lo` / `AQHI_hi` (Canadian formula) | lo: means (lower bound); hi: maxima (upper bound) |
| `AQHI_ln` (optional) | lognormal fit on median and IQR; run `check_lognormal_fit` first |

`lo`/`hi` bracket the unobservable hourly-based value. For the flags, `*_lo_ge4` is a certain exceedance and `*_hi_ge4` a possible one.
On daily-resolution PM rows (min = max) the intraday statistics are set to NaN, so peaks and spreads only use hourly rows.

In [19]:
import re
import warnings

RELIABILITY_DIR = PROJECT_DIR / "1_reliability"

# ---------------------------------------------------------------------------
# Constants and configuration
# ---------------------------------------------------------------------------
AQ_STATS = ("min", "q1", "mean", "med", "q3", "max")
# NH3 is left out on purpose: a single station / a single usable year. PM2.5 -> PM25 in column names.
POLLUTANTS = ("CO", "NO", "NO2", "O3", "PM10", "PM25", "SO2")
UNITS = {p: ("mg/m3" if p == "CO" else "ug/m3") for p in POLLUTANTS}  # CO is in mg/m3
PM_POLLUTANTS = ("PM10", "PM25")  # PM series may be daily already (min == max == mean) -> no intraday information

# Mass <-> mixing-ratio conversion at the EU reference conditions (293.15 K, 101.325 kPa)
MOLAR_VOLUME = 24.055  # L/mol
MOLAR_MASS = {"NO": 30.006, "NO2": 46.006, "O3": 47.998}  # g/mol
NOX_FACTOR = MOLAR_MASS["NO2"] / MOLAR_MASS["NO"]  # ~1.533: NOx expressed as NO2 equivalent


def ug_m3_to_ppb(x, species):
    return x * MOLAR_VOLUME / MOLAR_MASS[species]


# Daily exceedance indicators: name -> (pollutant, daily statistic, threshold, note)
# "mean" = daily mean (24-h definitions), "max" = daily maximum (hourly definitions).
# Edit freely: every entry becomes a 0/1/NaN column in compute_daily_indices.
THRESHOLDS = {
    # EU 2008/50/EC 24-h limit values (daily mean)
    "EX_PM10_EU50":  ("PM10", "mean", 50,  "EU 24-h limit (35 exceedances/yr allowed)"),
    "EX_SO2_EU125":  ("SO2",  "mean", 125, "EU 24-h limit (3 exceedances/yr allowed)"),
    # EU 2008/50/EC hourly thresholds, tested on the daily maximum
    "EX_NO2_H200":   ("NO2",  "max",  200, "EU hourly limit (18 exceedances/yr allowed)"),
    "EX_SO2_H350":   ("SO2",  "max",  350, "EU hourly limit (24 exceedances/yr allowed)"),
    "EX_O3_H180":    ("O3",   "max",  180, "EU information threshold (hourly)"),
    "EX_O3_H240":    ("O3",   "max",  240, "EU alert threshold (hourly)"),
    # WHO 2021 24-h guidelines (daily mean; CO in mg/m3)
    "EX_PM25_WHO15": ("PM25", "mean", 15,  "WHO 2021 24-h guideline"),
    "EX_PM10_WHO45": ("PM10", "mean", 45,  "WHO 2021 24-h guideline"),
    "EX_NO2_WHO25":  ("NO2",  "mean", 25,  "WHO 2021 24-h guideline"),
    "EX_SO2_WHO40":  ("SO2",  "mean", 40,  "WHO 2021 24-h guideline"),
    "EX_CO_WHO4":    ("CO",   "mean", 4,   "WHO 2021 24-h guideline (mg/m3)"),
}
# Not included on purpose: an hourly CO > 10 mg/m3 flag (spikes survive the source QC cap of 100 mg/m3,
# and the legal 8-h metric cannot be rebuilt from daily statistics).

# EEA AQI class upper bounds (classes 1..6 = Good, Fair, Moderate, Poor, Very poor, Extremely poor), ug/m3.
EEA_AQI_UPPER = {
    "PM25": (5, 15, 50, 90, 140),
    "PM10": (15, 45, 120, 195, 270),
    "NO2":  (10, 25, 60, 100, 150),
    "O3":   (60, 100, 120, 160, 180),
    "SO2":  (20, 40, 125, 190, 275),
}

# Canadian AQHI: 10/10.4 * 100 * sum_j (exp(k_j * C_j) - 1); O3 and NO2 in ppb, PM2.5 in ug/m3
AQHI_K = {"O3": 0.000537, "NO2": 0.000871, "PM25": 0.000487}
AQHI_SCALE = 10 / 10.4 * 100


In [20]:
_AQ_RE = re.compile(r"^AQ_(min|q1|mean|med|q3|max)_(.+)$")
INTRADAY_STATS = ("min", "q1", "med", "q3", "max")


def standardize_columns(df):
    """Rename AQ_<stat>_<pollutant> columns so that the pollutant tag is alphanumeric
    (e.g. PM2.5 -> PM25). Other columns are untouched."""
    rename = {}
    for col in df.columns:
        m = _AQ_RE.match(col)
        if m:
            tag = re.sub(r"[^0-9A-Za-z]", "", m.group(2)).upper()
            rename[col] = f"AQ_{m.group(1)}_{tag}"
    if not rename:
        raise KeyError("No AQ_<stat>_<pollutant> columns found")
    return df.rename(columns=rename)


def check_stat_order(df, tol=1e-6):
    """Sanity check of the daily statistics: min <= q1 <= med <= q3 <= max and min <= mean <= max."""
    rows = []
    for pol in POLLUTANTS:
        cols = [f"AQ_{s}_{pol}" for s in INTRADAY_STATS]
        if not all(c in df.columns for c in cols) or f"AQ_mean_{pol}" not in df.columns:
            continue
        v = df[cols].to_numpy(dtype=float)
        mean = df[f"AQ_mean_{pol}"].to_numpy(dtype=float)
        ok = ~np.isnan(v).any(axis=1)
        bad_order = (np.diff(v, axis=1) < -tol).any(axis=1) & ok
        bad_mean = ((mean < v[:, 0] - tol) | (mean > v[:, 4] + tol)) & ok
        rows.append({
            "pollutant": pol,
            "complete_rows": int(ok.sum()),
            "quantile_order_violations": int(bad_order.sum()),
            "mean_outside_min_max": int(bad_mean.sum()),
        })
    return pd.DataFrame(rows).set_index("pollutant")


def check_lognormal_fit(df, pollutants=("NO2", "O3", "SO2", "CO", "PM10", "PM25")):
    """Diagnostic for AQHI_ln: if the hourly values of a day were lognormal with the observed median and
    IQR, the implied daily mean would be med * exp(sigma^2 / 2), sigma = ln(q3/q1) / 1.349.
    Compares it with the reported daily mean on rows with intraday information (q3 > q1).
    Small relative errors support using AQHI_ln; large or skewed ones do not."""
    rows = []
    for p in pollutants:
        cols = [f"AQ_{s}_{p}" for s in ("q1", "med", "q3", "mean")]
        if not all(c in df.columns for c in cols):
            continue
        q1, med, q3, mean = (df[c].astype(float) for c in cols)
        ok = (q1 > 0) & (q3 > q1) & (med > 0) & (mean > 0)
        sigma = np.log(q3 / q1) / 1.349
        rel = (med * np.exp(sigma ** 2 / 2) / mean - 1)[ok]
        rows.append({"pollutant": p, "rows": int(ok.sum()), "rel_error_median": rel.median(),
                     "abs_rel_error_median": rel.abs().median(),
                     "rel_error_p10": rel.quantile(0.1), "rel_error_p90": rel.quantile(0.9)})
    return pd.DataFrame(rows).set_index("pollutant")


def _aqi_class(x, upper_bounds):
    """EEA class 1..6 (float, NaN where x is NaN). Class k means x <= upper_bounds[k-1]."""
    arr = x.to_numpy(dtype=float)
    cls = np.searchsorted(np.asarray(upper_bounds, dtype=float), arr, side="left") + 1
    cls = cls.astype("float32")
    cls[np.isnan(arr)] = np.nan
    return pd.Series(cls, index=x.index)


def _aqhi(o3_ppb, no2_ppb, pm25):
    return AQHI_SCALE * (np.expm1(AQHI_K["O3"] * o3_ppb) + np.expm1(AQHI_K["NO2"] * no2_ppb)
                         + np.expm1(AQHI_K["PM25"] * pm25))


_GH_X, _GH_W = np.polynomial.hermite.hermgauss(24)


def _aqhi_term_lognormal(k, conv, med, q1, q3, lo, hi, chunk=250_000):
    """E[expm1(k * conv * X)] for X lognormal (median `med`, sigma = ln(q3/q1)/1.349), X clipped to [lo, hi]
    (the day's min and max). NaN where the lognormal cannot be fitted (med <= 0 or q1 <= 0)."""
    med, q1, q3, lo, hi = (np.asarray(a, dtype=float) for a in (med, q1, q3, lo, hi))
    lo = np.where(np.isnan(lo), 0.0, lo)
    hi = np.where(np.isnan(hi), np.inf, hi)
    out = np.full(len(med), np.nan)
    for s in range(0, len(med), chunk):
        sl = slice(s, s + chunk)
        m, a, b = med[sl], q1[sl], q3[sl]
        ok = (m > 0) & (a > 0) & (b >= a)
        mu = np.log(np.where(ok, m, 1.0))
        sig = np.where(ok, np.log(np.where(ok, b, 1.0) / np.where(ok, a, 1.0)) / 1.349, 0.0)
        x = np.exp(mu[:, None] + sig[:, None] * np.sqrt(2.0) * _GH_X[None, :])
        x = np.clip(x, lo[sl][:, None], np.maximum(hi[sl][:, None], lo[sl][:, None]))
        val = (np.expm1(k * conv * x) @ _GH_W) / np.sqrt(np.pi)
        out[sl] = np.where(ok, val, np.nan)
    return out


def compute_daily_indices(df, thresholds=None, aqi_min_pollutants=2, rpm_max=1.0, aqhi_lognormal=False):
    """Station-day indices from the daily statistics. Returns ONLY the new columns, aligned to df.index.

      <P>_iqr           intraday spread q3 - q1 (NaN on daily-resolution PM rows, where min == max)
      PM10/PM25_daily_only   1 if the PM row is daily-resolution (no intraday information), else 0
      RPM               PM2.5 / PM10 from daily means; NaN if PM10 <= 0 or ratio > rpm_max
      NOx               1.533*NO + NO2 (NO2-equivalent mass, ug/m3) from daily means
      NO2_NOx           NO2 share of NOx (mass-equivalent basis; low = fresh combustion, high = aged air)
      Ox_ppb            O3 + NO2 in ppb (daily means converted to mixing ratios first)
      AQI_lo / AQI_hi   EEA-inspired class 1-6. lo: all daily means. hi: PM on the 24-h mean, NO2/O3/SO2 on the
                        daily max (= worst-hour class of the day). Needs >= aqi_min_pollutants pollutants.
      AQI_hi_driver, AQI_hi_n, AQI_hi_drv_<P>   pollutant setting AQI_hi, number of pollutants used, 0/1 dummies
      AQHI_lo / AQHI_hi Canadian AQHI formula on daily means (lower bound) / on daily maxima (upper bound).
                        AQHI is a sum of convex terms, so by Jensen's inequality AQHI(daily means) <= the true daily
                        mean of the hourly AQHI <= its daily max <= AQHI(daily maxima). Caveat: PM2.5 on
                        daily-resolution stations has no hourly maximum, so its mean is used there and the PM term of
                        AQHI_hi is then not a strict upper bound (see PM25_daily_only).
      AQHI_ln           (only if aqhi_lognormal=True) estimate of the daily mean of the hourly AQHI, assuming
                        lognormal hourly values fitted on median and IQR. Run check_lognormal_fit() first.
      EX_*              0/1/NaN exceedance flags from `thresholds`, plus AQI/AQHI >= 4 for lo (certain) and hi (possible)
    All are proxies built from daily statistics, not official indices.
    """
    thresholds = THRESHOLDS if thresholds is None else thresholds
    out = {}

    def get(stat, pol):
        name = f"AQ_{stat}_{pol}"
        return df[name] if name in df.columns else None

    def flag(series, limit):
        return (series >= limit).astype("float32").where(series.notna())

    # --- 1. intraday spread and daily-resolution PM rows -------------------------------------
    for pol in POLLUTANTS:
        mn, mx, q1, q3 = get("min", pol), get("max", pol), get("q1", pol), get("q3", pol)
        daily_only = None
        if pol in PM_POLLUTANTS and mn is not None and mx is not None:
            daily_only = mn.eq(mx)
            out[f"{pol}_daily_only"] = daily_only.astype("float32").where(mn.notna() & mx.notna())
        if q1 is not None and q3 is not None:
            iqr = (q3 - q1).where(lambda s: s >= 0)
            if daily_only is not None:
                iqr = iqr.mask(daily_only)
            out[f"{pol}_iqr"] = iqr.astype("float32")

    # --- 2. ratios and chemical indicators (daily means: additive, same station-day) --------------
    pm25, pm10 = get("mean", "PM25"), get("mean", "PM10")
    if pm25 is not None and pm10 is not None:
        rpm = pm25 / pm10.where(pm10 > 0)
        out["RPM"] = rpm.where(rpm <= rpm_max).astype("float32")
    no, no2, o3 = get("mean", "NO"), get("mean", "NO2"), get("mean", "O3")
    if no is not None and no2 is not None:
        nox = NOX_FACTOR * no + no2
        out["NOx"] = nox.astype("float32")
        out["NO2_NOx"] = (no2 / nox.where(nox > 0)).astype("float32")
    if o3 is not None and no2 is not None:
        out["Ox_ppb"] = (ug_m3_to_ppb(o3, "O3") + ug_m3_to_ppb(no2, "NO2")).astype("float32")

    # --- 3. EEA-inspired AQI proxies ----------------------------------------------------------
    for label, gas_stat in (("lo", "mean"), ("hi", "max")):
        subs = {}
        for pol, bounds in EEA_AQI_UPPER.items():
            s = get("mean" if pol in PM_POLLUTANTS else gas_stat, pol)
            if s is not None:
                subs[pol] = _aqi_class(s, bounds)
        if not subs:
            continue
        sub = pd.DataFrame(subs)
        n = sub.notna().sum(axis=1)
        valid = n >= aqi_min_pollutants
        aqi = sub.max(axis=1).where(valid)
        out[f"AQI_{label}"] = aqi.astype("float32")
        out[f"EX_AQI_{label}_ge4"] = flag(aqi, 4)
        if label == "hi":
            out["AQI_hi_n"] = n.astype("float32")
            driver = sub.fillna(-1).idxmax(axis=1).where(valid)  # ties: first of PM25, PM10, NO2, O3, SO2
            out["AQI_hi_driver"] = driver.astype("category")
            for pol in subs:
                out[f"AQI_hi_drv_{pol}"] = (driver == pol).astype("float32").where(driver.notna())

    # --- 4. AQHI proxies ----------------------------------------------------------------------
    pm25m = get("mean", "PM25")
    if no2 is not None and o3 is not None and pm25m is not None:
        lo = _aqhi(ug_m3_to_ppb(o3, "O3"), ug_m3_to_ppb(no2, "NO2"), pm25m)
        out["AQHI_lo"] = lo.astype("float32")
        out["EX_AQHI_lo_ge4"] = flag(lo, 4)
        o3x, no2x, pmx = get("max", "O3"), get("max", "NO2"), get("max", "PM25")
        if o3x is not None and no2x is not None:
            pmx = pm25m if pmx is None else pmx.fillna(pm25m)
            hi = _aqhi(ug_m3_to_ppb(o3x, "O3"), ug_m3_to_ppb(no2x, "NO2"), pmx)
            out["AQHI_hi"] = hi.astype("float32")
            out["EX_AQHI_hi_ge4"] = flag(hi, 4)
        if aqhi_lognormal:
            terms = []
            for pol, conv in (("O3", MOLAR_VOLUME / MOLAR_MASS["O3"]), ("NO2", MOLAR_VOLUME / MOLAR_MASS["NO2"]),
                              ("PM25", 1.0)):
                terms.append(_aqhi_term_lognormal(
                    AQHI_K[pol], conv, get("med", pol).to_numpy(), get("q1", pol).to_numpy(),
                    get("q3", pol).to_numpy(), get("min", pol).to_numpy(), get("max", pol).to_numpy()))
            out["AQHI_ln"] = pd.Series((AQHI_SCALE * (terms[0] + terms[1] + terms[2])).astype("float32"), index=df.index)

    # --- 5. exceedance flags --------------------------------------------------------------------
    for name, (pol, stat, limit, _note) in thresholds.items():
        s = get(stat, pol)
        if s is not None:
            out[name] = (s > limit).astype("float32").where(s.notna())

    return pd.DataFrame(out, index=df.index)


def build_daily_dataset(df, date_col="time", station_col="AirQualityStation", downcast=True, **index_kwargs):
    """standardize columns + keep only station/date/AQ_*/CL_* (NH3 dropped) + append daily indices.
    On daily-resolution PM rows the intraday statistics (min, q1, med, q3, max) are set to NaN afterwards,
    so that peak and spread statistics only use rows that really have hourly information."""
    df = standardize_columns(df)
    aq = [f"AQ_{s}_{p}" for p in POLLUTANTS for s in AQ_STATS if f"AQ_{s}_{p}" in df.columns]
    cl = [c for c in df.columns if c.startswith("CL_")]
    keep = [station_col, date_col] + aq + cl
    df = df[keep].copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="raise")
    if downcast:
        df[aq + cl] = df[aq + cl].astype("float32")
    df = df.join(compute_daily_indices(df, **index_kwargs))
    for pol in PM_POLLUTANTS:
        if f"{pol}_daily_only" in df.columns:
            cols = [f"AQ_{s}_{pol}" for s in INTRADAY_STATS if f"AQ_{s}_{pol}" in df.columns]
            df.loc[df[f"{pol}_daily_only"] == 1, cols] = np.nan
    return df


DAILY_INDEX_COLS = ("RPM", "NOx", "NO2_NOx", "Ox_ppb", "AQI_lo", "AQI_hi", "AQI_hi_driver",
                    "AQHI_lo", "AQHI_hi", "AQHI_ln")


def daily_indices_table(df_daily, date_col="time", station_col="AirQualityStation"):
    """Station-day table with the indices and exceedance flags only (no concentrations, no spread statistics)."""
    cols = [c for c in DAILY_INDEX_COLS if c in df_daily.columns]
    cols += [c for c in df_daily.columns if c.startswith("EX_")]
    return df_daily[[station_col, date_col] + cols]


## 4. Aggregation (week / month / season / year)

Returns two tables with identical keys and row order: the statistics (`1_processed/aggregated_<freq>.csv`) and the data-quality counts (`1_reliability/reliability_<freq>.csv`). Keys: `week` (Monday) | `year`, `month` | `year`, `season` | `year`. Winter is labelled with the year of its December. Periods not entirely inside the dataset (first week, winter 2012–13, winter 2023–24) are dropped; sparse station-periods are masked below `min_coverage` (default 0.5).

In [21]:
DERIVED_COLS = ("RPM", "NOx", "NO2_NOx", "Ox_ppb", "AQI_lo", "AQI_hi", "AQHI_lo", "AQHI_hi", "AQHI_ln")
CLIMATE_MEAN_STD = ("CL_t2m", "CL_rh", "CL_windspeed", "CL_blh", "CL_ssr")  # CL_ssr is a daily maximum
CLIMATE_MEAN_ONLY = ("CL_lai_hv", "CL_lai_lv")
WIND_SECTORS = {1: "N", 2: "E", 3: "S", 4: "W"}
WET_DAY_M = 0.001  # 1 mm of precipitation, in metres (CL_tp unit)
SEASON_NAMES = {12: "Winter", 3: "Spring", 6: "Summer", 9: "Autumn"}
_STEP = {"week": pd.Timedelta(days=7), "month": pd.DateOffset(months=1),
         "season": pd.DateOffset(months=3), "year": pd.DateOffset(years=1)}
MIN_COVERAGE = 0.33


def _period_start(dates, frequency):
    if frequency == "week":  # Monday-Sunday
        return dates.dt.to_period("W-SUN").dt.start_time
    if frequency == "month":
        return dates.dt.to_period("M").dt.start_time
    if frequency == "year":
        return dates.dt.to_period("Y").dt.start_time
    start_month = dates.dt.month.map(
        {12: 12, 1: 12, 2: 12, 3: 3, 4: 3, 5: 3, 6: 6, 7: 6, 8: 6, 9: 9, 10: 9, 11: 9}
    )
    start_year = dates.dt.year - (dates.dt.month <= 2).astype(int)  # Jan-Feb belong to the winter that began in December
    return pd.to_datetime({"year": start_year, "month": start_month, "day": 1})


def _period_span(starts, frequency):
    """(last day, length in days) of the full calendar period beginning at each start date."""
    starts = pd.DatetimeIndex(starts)
    uniq = starts.unique()
    nxt = pd.DatetimeIndex([s + _STEP[frequency] for s in uniq])
    table = pd.DataFrame({"end": nxt - pd.Timedelta(days=1), "days": (nxt - uniq).days}, index=uniq).reindex(starts)
    return pd.DatetimeIndex(table["end"]), table["days"].to_numpy(dtype=float)


def _time_columns(period, frequency):
    """Key columns that replace the period start date: week | year, month | year, season | year."""
    if frequency == "week":
        return {"week": period}
    if frequency == "month":
        return {"year": period.year, "month": period.month}
    if frequency == "season":  # winter is labelled with the year of its December
        return {"year": period.year, "season": period.month.map(SEASON_NAMES)}
    return {"year": period.year}


def aggregate_measurements(df, frequency, date_col="time", group_cols=("AirQualityStation",),
                           station_col="AirQualityStation", min_coverage=MIN_COVERAGE, drop_partial_periods=True):
    """Aggregate the daily dataset built by build_daily_dataset() to week / month / season / year.
    Returns (main, reliability): two frames with the same keys and row order.

    Key columns: week (Monday) | year, month | year, season (winter = year of its December) | year.

    main
      pollutant P   P_mean / P_median / P_std / P_p95    over the DAILY MEANS (std = day-to-day variability)
                    P_max_mean / P_max_peak               mean / maximum of the daily maxima
                    P_iqr_mean                            mean intraday spread (daily q3 - q1)
      derived       RPM, NOx, NO2_NOx, Ox_ppb, AQI_lo/hi, AQHI_lo/hi(/ln)   mean / median / std / p95
      AQI_hi_drv_<P>_share   share of valid days in which pollutant P sets AQI_hi
      EX_* flags    <flag>_n, <flag>_pct                  days above the threshold / % of valid days
      climate       t2m, rh, windspeed, blh, ssr: mean, std; CL_tp_sum and CL_tp_wet_days (>= 1 mm);
                    CL_winddir_<sector>_share and CL_winddir_mode (1=N, 2=E, 3=S, 4=W); LAI: mean
    reliability
      n_days_expected, <variable>_n_valid (valid days), <PM>_n_daily_only (days with daily-resolution PM only),
      and for frequency="year" <variable>_n_months (months with at least one valid day).

    min_coverage: when grouping by station, all statistics of a variable are set to NaN if valid days / calendar
    days of the period < min_coverage (0 disables). Climate is complete and never masked. When pooling across
    stations (station_col not in group_cols) nothing is masked, because coverage is not comparable across
    stations that do not measure the variable; reliability then also reports n_stations.
    drop_partial_periods: drop periods that are not entirely inside the dataset's date range (first week,
    first and last winter), independently of data coverage.
    """
    if frequency not in {"week", "month", "season", "year"}:
        raise ValueError("frequency must be one of ['month', 'season', 'week', 'year']")
    group_cols = list(group_cols)
    missing = [c for c in [date_col, *group_cols] if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    dates = pd.to_datetime(df[date_col], errors="raise")
    if dates.isna().any():
        raise ValueError(f"{date_col!r} contains missing dates")
    period = _period_start(dates, frequency)
    period.name = "period"
    grouping = [df[c] for c in group_cols] + [period]
    g = df.groupby(grouping, sort=True, observed=True)
    pooled = station_col not in group_cols

    present = [p for p in POLLUTANTS if f"AQ_mean_{p}" in df.columns]
    derived = [c for c in DERIVED_COLS if c in df.columns]
    flags = [c for c in df.columns if c.startswith("EX_")]
    drivers = [c for c in df.columns if c.startswith("AQI_hi_drv_")]
    climate_ms = [c for c in CLIMATE_MEAN_STD if c in df.columns]
    climate_m = [c for c in CLIMATE_MEAN_ONLY if c in df.columns]

    pieces, groups, counts = [], {}, {}

    def collect(frame, label_of, stat):
        renamed = frame.rename(columns=lambda c: f"{label_of(c)}_{stat}")
        pieces.append(renamed)
        for old, new in zip(frame.columns, renamed.columns):
            groups.setdefault(label_of(old), []).append(new)

    cut_aq = lambda c: c.split("_", 2)[2]  # AQ_<stat>_<P> -> P
    cut_iqr = lambda c: c[:-4]             # <P>_iqr -> P
    same = lambda c: c

    # pollutants + derived indices: level and day-to-day variability of the daily value
    level_cols = [f"AQ_mean_{p}" for p in present] + derived
    label_level = lambda c: cut_aq(c) if c.startswith("AQ_mean_") else c
    gl = g[level_cols]
    collect(gl.mean(), label_level, "mean")
    collect(gl.median(), label_level, "median")
    collect(gl.std(), label_level, "std")
    collect(gl.quantile(0.95), label_level, "p95")
    for c, s in gl.count().items():
        counts[label_level(c)] = s

    # pollutants: peaks and intraday spread (hourly-resolution rows only)
    max_cols = [f"AQ_max_{p}" for p in present if f"AQ_max_{p}" in df.columns]
    if max_cols:
        collect(g[max_cols].mean(), cut_aq, "max_mean")
        collect(g[max_cols].max(), cut_aq, "max_peak")
    iqr_cols = [f"{p}_iqr" for p in present if f"{p}_iqr" in df.columns]
    if iqr_cols:
        collect(g[iqr_cols].mean(), cut_iqr, "iqr_mean")

    # AQI driver shares
    if drivers:
        share = g[drivers].mean()
        share.columns = [f"{c}_share" for c in drivers]
        pieces.append(share)
        groups.setdefault("AQI_hi", []).extend(share.columns)

    # exceedance flags
    if flags:
        gf = g[flags]
        n_exc, n_val = gf.sum(min_count=1), gf.count()
        collect(n_exc, same, "n")
        collect(100.0 * n_exc / n_val.where(n_val > 0), same, "pct")
        for c in flags:
            counts[c] = n_val[c]

    # climate (complete: no counts, no masking)
    if climate_ms:
        collect(g[climate_ms].mean(), same, "mean")
        collect(g[climate_ms].std(), same, "std")
    if climate_m:
        collect(g[climate_m].mean(), same, "mean")
    extra = {}
    if "CL_tp" in df.columns:
        extra["CL_tp_wet"] = (df["CL_tp"] >= WET_DAY_M).astype("float32").where(df["CL_tp"].notna())
    if "CL_winddir" in df.columns:
        for code, sector in WIND_SECTORS.items():
            extra[f"CL_winddir_{sector}"] = (df["CL_winddir"] == code).astype("float32").where(df["CL_winddir"].notna())
    if extra:
        ge = pd.DataFrame(extra, index=df.index).groupby(grouping, sort=True, observed=True)
        if "CL_tp" in df.columns:
            pieces.append(g["CL_tp"].sum(min_count=1).rename("CL_tp_sum").to_frame())
            pieces.append(ge["CL_tp_wet"].sum(min_count=1).rename("CL_tp_wet_days").to_frame())
        if "CL_winddir" in df.columns:
            sector_cols = [f"CL_winddir_{s}" for s in WIND_SECTORS.values()]
            share = ge[sector_cols].mean()
            share.columns = [f"{c}_share" for c in sector_cols]
            mode = pd.Series(share.fillna(-1).to_numpy().argmax(axis=1) + 1, index=share.index, dtype="float64")
            pieces.extend([share, mode.where(share.notna().any(axis=1)).rename("CL_winddir_mode").to_frame()])

    result = pd.concat(pieces, axis=1)

    # periods not entirely inside the dataset (first week, first and last winter)
    periods = result.index.get_level_values("period")
    if drop_partial_periods:
        ends, _ = _period_span(periods, frequency)
        result = result.loc[(periods >= dates.min()) & (ends <= dates.max())]
    periods = result.index.get_level_values("period")
    _, expected = _period_span(periods, frequency)

    # reliability + masking of sparse periods
    rel = {"n_days_expected": pd.Series(expected, index=result.index)}
    if pooled and station_col in df.columns:
        rel["n_stations"] = g[station_col].nunique().reindex(result.index)
    for label, series in counts.items():
        n_valid = series.reindex(result.index)
        rel[f"{label}_n_valid"] = n_valid.astype("int32")
        if min_coverage > 0 and not pooled:
            bad = (n_valid / expected < min_coverage).to_numpy()
            if bad.any():
                result.loc[bad, groups[label]] = np.nan
    daily_only = [f"{p}_daily_only" for p in PM_POLLUTANTS if f"{p}_daily_only" in df.columns]
    if daily_only:
        n_do = g[daily_only].sum(min_count=1).reindex(result.index)
        n_do.columns = [c.replace("_daily_only", "_n_daily_only") for c in daily_only]
        rel.update({c: n_do[c] for c in n_do.columns})
    if frequency == "year":  # months with at least one valid day, to spot seasonally unbalanced years
        month = dates.dt.month
        month.name = "_month"
        gm = df[level_cols].groupby(grouping + [month], sort=False, observed=True).count()
        months = (gm > 0).groupby(level=list(range(len(grouping)))).sum().reindex(result.index)
        for c in level_cols:
            rel[f"{label_level(c)}_n_months"] = months[c].astype("int16")

    keys = result.index.to_frame(index=False)
    period_idx = pd.DatetimeIndex(keys.pop("period"))
    for name, values in _time_columns(period_idx, frequency).items():
        keys[name] = values
    main = pd.concat([keys, result.reset_index(drop=True)], axis=1).copy()
    reliability = pd.concat([keys, pd.DataFrame(rel).reset_index(drop=True)], axis=1).copy()
    return main, reliability


## 5. Run

In [22]:
df_daily = build_daily_dataset(df_measurements)
check_stat_order(df_daily)  # min <= q1 <= med <= q3 <= max, min <= mean <= max

,complete_rows,quantile_order_violations,mean_outside_min_max
pollutant,,,
CO,708481,0,0
NO,469551,0,0
NO2,2089814,0,0
O3,1143121,0,0
PM10,252421,0,0
PM25,104024,0,0
SO2,700375,0,0


In [23]:
# Optional: is a lognormal day a reasonable description? Small errors support AQHI_ln.
# If so, rebuild with: df_daily = build_daily_dataset(df_measurements, aqhi_lognormal=True)
check_lognormal_fit(df_daily)

,rows,rel_error_median,abs_rel_error_median,rel_error_p10,rel_error_p90
pollutant,,,,,
NO2,2080680,0.034663,0.083370,-0.107680,0.333984
O3,1139883,0.068957,0.094854,-0.045450,0.699678
SO2,626707,0.004078,0.041681,-0.108241,0.219266
CO,667259,-0.003525,0.047037,-0.107768,0.158874
PM10,230212,0.022846,0.060169,-0.083950,0.231305
PM25,92306,0.025489,0.064676,-0.091007,0.248994


In [24]:
# Daily file: indices and exceedance flags only
daily_table = daily_indices_table(df_daily)
daily_table.to_csv(PROCESSED_DIR / "daily_indices.csv.gz", index=False)
print(daily_table.shape)

(2988648, 26)


In [25]:
RELIABILITY_DIR.mkdir(parents=True, exist_ok=True)
aggregated = {}
for frequency in ("week", "month", "season", "year"):
    main, reliability = aggregate_measurements(df_daily, frequency)
    main.to_csv(PROCESSED_DIR / f"aggregated_{frequency}.csv", index=False)
    reliability.to_csv(RELIABILITY_DIR / f"reliability_{frequency}.csv", index=False)
    aggregated[frequency] = main
    print(frequency, main.shape, reliability.shape)

week (426312, 137) (426312, 35)
month (98208, 138) (98208, 36)
season (31992, 138) (31992, 36)
year (8184, 137) (8184, 50)


In [26]:
# Example: stricter data capture (>= 90% valid days) from the reliability file, then annual medians by station type
annual = aggregated["year"].join(
    df_stations[["AirQualityStationType", "AirQualityStationArea", "Regione"]], on="AirQualityStation"
)
rel = pd.read_csv(RELIABILITY_DIR / "reliability_year.csv")  # same row order as the main file
strict = (rel["NO2_n_valid"] / rel["n_days_expected"] >= 0.9).to_numpy()
annual[strict].groupby("AirQualityStationType")[["NO2_mean", "EX_NO2_WHO25_pct", "AQI_hi_mean"]].median()

,NO2_mean,EX_NO2_WHO25_pct,AQI_hi_mean
AirQualityStationType,,,
background,18.392580,23.285627,3.106557
industrial,16.377722,13.202247,2.861110
traffic,29.927341,59.077810,3.385041
